# Combine IHME GBD 2023 PM2.5 annual GeoTIFFs into a single time-series NetCDF.

Handles the fact that the source rasters aren't all on the same grid:
- 1990-2005 files span 70N to 55S
- 2006-2009 files span 68N to 55S (missing the northern 2 degrees)

All years are placed on a common target grid (union extent), with NaN
filling any area a given year doesn't cover.

In [ ]:
import glob
import os
import re
import numpy as np
import rasterio
import xarray as xr
import config
from utils.utils import require_dir
import pathlib

In [ ]:
DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "PM2.5_obs")
out_file = "IHME_GBD_2023_PM25_timeseries.nc"
out_path = os.path.join(DIR, out_file)

RES = 0.1  # degrees, native resolution of the source rasters

files = sorted(glob.glob(os.path.join(DIR, "*.TIF")))

# --- Pass 1: read each file's extent/metadata and find the union grid ---
metas = []
top_max, bottom_min = -1e9, 1e9
left_min, right_max = 1e9, -1e9

for f in files:
    with rasterio.open(f) as src:
        t = src.transform
        h, w = src.shape
        top, left = t.f, t.c
        bottom = top - h * RES
        right = left + w * RES
        metas.append((f, h, w, top, left, bottom, right, src.nodata))
        top_max = max(top_max, top)
        bottom_min = min(bottom_min, bottom)
        left_min = min(left_min, left)
        right_max = max(right_max, right)

n_rows = round((top_max - bottom_min) / RES)
n_cols = round((right_max - left_min) / RES)

lon_target = left_min + (np.arange(n_cols) + 0.5) * RES
lat_target = top_max - (np.arange(n_rows) + 0.5) * RES

# --- Pass 2: read each raster, mask nodata, and place onto the common grid ---
years, data_list = [], []

for f, h, w, top, left, bottom, right, nodata in metas:
    year = int(re.search(r"PM_(\d{4})_Y", os.path.basename(f)).group(1))
    years.append(year)

    with rasterio.open(f) as src:
        arr = src.read(1).astype("float32")
        if src.nodata is not None:
            arr = np.where(arr == src.nodata, np.nan, arr)

    row_off = round((top_max - top) / RES)
    col_off = round((left - left_min) / RES)

    full = np.full((n_rows, n_cols), np.nan, dtype="float32")
    full[row_off:row_off + h, col_off:col_off + w] = arr
    data_list.append(full)

data = np.stack(data_list, axis=0)  # (time, lat, lon)

# --- Build the xarray Dataset ---
ds = xr.Dataset(
    {
        "pm25": (
            ("time", "lat", "lon"),
            data,
            {
                "long_name": "PM2.5 concentration",
                "units": "micrograms per cubic meter",
                "source": "IHME GBD 2023 Air Pollution estimates",
            },
        )
    },
    coords={
        "time": np.array([f"{y}-01-01" for y in years], dtype="datetime64[ns]"),
        "lat": ("lat", lat_target.astype("float32"),
                {"units": "degrees_north", "long_name": "latitude"}),
        "lon": ("lon", lon_target.astype("float32"),
                {"units": "degrees_east", "long_name": "longitude"}),
    },
    attrs={
        "title": "IHME GBD 2023 Air Pollution PM2.5 Time Series",
        "description": (
            "Time series of global gridded PM2.5 concentration estimates, "
            "compiled from individual annual GeoTIFFs (IHME GBD 2023). "
            "Note: source rasters for 2006-2009 cover a slightly smaller "
            "latitude extent (68N to 55S) than 1990-2005 (70N to 55S); "
            "missing northern rows for those years are filled with NaN."
        ),
        "years_included": ", ".join(str(y) for y in years),
        "resolution_degrees": RES,
        "crs": "EPSG:4326 (WGS84, assumed - not embedded in source TIFFs)",
        "history": (
            "Created by stacking annual GeoTIFF rasters onto a common grid "
            "and combining into NetCDF with xarray/rasterio."
        ),
    },
)

encoding = {"pm25": {"zlib": True, "complevel": 4}}
ds.to_netcdf(out_path, encoding=encoding)
print(f"Saved {out_path}")
print(ds)